In [2]:
import os

In [3]:
%pwd

'd:\\PredictBot-Score-MLOps\\research'

In [4]:
os.chdir('..')

In [5]:
%pwd

'd:\\PredictBot-Score-MLOps'

In [6]:
from dataclasses import dataclass
from pathlib import Path
from src.predictor_bot_score.config.configuration import yaml_load , create_directories
from src.predictor_bot_score.logger import logger
from src.predictor_bot_score.constants import CONFIG_PATH
import os
import pandas as pd
import glob
import mlflow
import lightgbm
import pickle
from datetime import datetime
import sqlite3

In [7]:
@dataclass(frozen=True)
class ModelTrainingConfig:
    train_data_path      : Path
    val_data_path        : Path
    test_data_path       : Path
    model_dir            : Path
    features             : list[str]
    target_column        : str
    baseline_mae         : float
    promotion_threshold  : float
    lgbm_params          : dict
    mlflow_experiment    : str
    mlflow_tracking_uri  : str

In [8]:
class config_manager:

    def __init__(self, config = CONFIG_PATH):

        self.config = yaml_load(config)
        
        create_directories([self.config.artifacts_root])

    def get_model_training_config(self) -> ModelTrainingConfig:

        config = self.config.model_training

        create_directories([config.model_dir])

        return ModelTrainingConfig(
            train_data_path     = Path(config.train_data_path),
            val_data_path       = Path(config.val_data_path),
            test_data_path      = Path(config.test_data_path),
            model_dir           = Path(config.model_dir),
            features            = list(config.features),
            target_column       = config.target_column,
            baseline_mae        = float(config.baseline_mae),
            promotion_threshold = float(config.promotion_threshold),
            lgbm_params         = dict(config.lgbm_params),
            mlflow_experiment   = config.mlflow.experiment_name,
            mlflow_tracking_uri = config.mlflow.tracking_uri
        )
        

In [ ]:
class Model_Building :

    def __init__(self, config : ModelTrainingConfig):
        self.config = config
        self.train , self.val = self._read_data()

    def _get_files(self , folder:Path ,prefix :str):
        files = glob.glob(os.path.join(folder ,f"{prefix}_*.csv"))
        if not files:
            raise FileNotFoundError(f"No {prefix} file found in {folder}")
        return max(files , key=os.path.getmtime)
    

    def _read_data(self):
        try:
            logger.info("=" * 50)
            logger.info("Reading train /  test splits")
            logger.info("=" * 50)

            train_file = self._get_files(self.config.train_data_path, "train")
            val_file   = self._get_files(self.config.val_data_path,   "val")

            train = pd.read_csv(train_file)
            val   = pd.read_csv(val_file)
            
            logger.info(f"Train : {len(train)} rows")
            logger.info(f"Val   : {len(val)} rows")

            return train, val

        except FileNotFoundError as e:
            logger.error(f"Split file not found: {e}")
            raise

        except Exception as e:
            logger.error(f"Failed to read splits: {str(e)}")
            raise

    def prepare_Data(self ,df :pd.DataFrame):
        try:
            X = df[self.config.features]
            y = df[self.config.target_column]

            return X ,y
        except Exception as e:
            raise e 
    
    def model_training(self):

        try:

            X_train, y_train = self.prepare_Data(self.train)
            X_val,   y_val   = self.prepare_Data(self.val)

            model = lightgbm.LGBMRegressor(**self.config.lgbm_params)
            model.fit(
                X_train , y_train,
                eval_set=[(X_val , y_val)],
                callbacks = [
                    lightgbm.early_stopping(100, verbose=False)
                   
                ]
            )

            logger.info(f"Best iteration : {model.best_iteration_}")
            logger.info("PASSED - Model trained")
            logger.info("-" * 50)

            return model

        except Exception as e:
            logger.error(f"Training failed: {str(e)}")
            raise
    
    def save_model(self,model):

        try:
            path = self.config.model_dir
            time_stamp = datetime.now().strftime("%Y_%m_%d_%H_%M")
            model_path = os.path.join(path,f"{time_stamp}.pkl")

            with open(model_path ,"wb") as f:
                pickle.dump(model , f)
        
            logger.info(f"Model saved ")
            logger.info("PASSED - Model saved")
            logger.info("-" * 50)

            return model_path

        except Exception as e:
            logger.error(f"Failed to save model: {str(e)}")
            raise

    # ── log to mlflow ─────────────────────────────────────
    def log_to_mlflow(self, model, model_path):
        try:
            logger.info("")
            logger.info("STEP 3 - LOGGING TO MLFLOW")
            logger.info("-" * 50)

            mlflow.set_tracking_uri(self.config.mlflow_tracking_uri)
            mlflow.set_experiment(self.config.mlflow_experiment)

            with mlflow.start_run():

                mlflow.log_params(self.config.lgbm_params)

                mlflow.set_tag("stage",      "Staging")
                mlflow.set_tag("trained_at", datetime.now().isoformat())
                mlflow.set_tag("model_path", model_path)


                mlflow.lightgbm.log_model(
                    model,
                    "model",
                    registered_model_name="Predicted_model_name"

                )
            logger.info("PASSED - MLflow logging complete")
            logger.info("-" * 50)

        except Exception as e:
            logger.error(f"MLflow logging failed: {str(e)}")
            raise

    def run(self):
        
        try:
            logger.info("=" * 50)
            logger.info("MODEL TRAINING PIPELINE STARTED")
            logger.info("=" * 50)

            model = self.model_training()
            model_save = self.save_model(model)
            self.log_to_mlflow(model , model_path=model_save)

            logger.info("=" * 50)
            logger.info("MODEL TRAINING PIPELINE COMPLETE")
            logger.info("=" * 50)

        except Exception as e:
            logger.error(f"Model training pipeline failed: {str(e)}")
            raise



            

   

In [10]:
xc = config_manager()
xc = xc.get_model_training_config()
xc = Model_Building(xc)
xc.run()


[2026-06-13 00:14:11,048: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-13 00:14:11,055: INFO: common: Directory created (or already exists) at: artifacts]
[2026-06-13 00:14:11,055: INFO: common: Directory created (or already exists) at: artifacts/models_directory/]
[2026-06-13 00:14:11,055: INFO: 15268866: ==================================================]


[2026-06-13 00:14:11,061: INFO: 15268866: Reading train /  test splits]
[2026-06-13 00:14:11,065: INFO: 15268866: ==================================================]
[2026-06-13 00:14:11,151: INFO: 15268866: Train : 28426 rows]
[2026-06-13 00:14:11,151: INFO: 15268866: Val   : 4061 rows]
[2026-06-13 00:14:11,155: INFO: 15268866: ==================================================]
[2026-06-13 00:14:11,155: INFO: 15268866: MODEL TRAINING PIPELINE STARTED]
[2026-06-13 00:14:11,157: INFO: 15268866: ==================================================]
[2026-06-13 00:14:18,878: INFO: 15268866: Best iteration : 1999]
[2026-06-13 00:14:18,878: INFO: 15268866: PASSED - Model trained]
[2026-06-13 00:14:18,878: INFO: 15268866: --------------------------------------------------]
[2026-06-13 00:14:19,240: INFO: 15268866: Model saved ]
[2026-06-13 00:14:19,240: INFO: 15268866: PASSED - Model saved]
[2026-06-13 00:14:19,240: INFO: 15268866: --------------------------------------------------]
[2026-06-

2026/06/13 00:14:21 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/13 00:14:21 INFO mlflow.store.db.utils: Updating database tables
2026/06/13 00:14:23 INFO mlflow.tracking.fluent: Experiment with name 'predict-bot-training' does not exist. Creating a new experiment.


[2026-06-13 00:14:24,121: ERROR: 15268866: MLflow logging failed: log_param() missing 1 required positional argument: 'value']
[2026-06-13 00:14:24,122: ERROR: 15268866: Model training pipeline failed: log_param() missing 1 required positional argument: 'value']


TypeError: log_param() missing 1 required positional argument: 'value'